In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "archive/src/experiment.py").is_file()
)
PROJECT_ROOT = ROOT = REPOSITORY_ROOT / "archive"
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))


# AGMFA: исправление attention features

Новый запуск: `resnet50_agmfa_robust_augmix_fixed_001`.
Три attention branches используют внутренние **512×16×16** выходы
`conv2 → bn2 → ReLU`, до `conv3`. Ручной residual forward проверяется
на точное численное совпадение с обычными bottleneck-блоками.
Fusion по-прежнему использует Res2(512), Res3(1024), финальный Res4(2048).
Robust AugMix, D4, AdamW, LR, 25 эпох, warmup3, frozen BN, partial и seed42 прежние.
Обучение выключено. Старый ноутбук с outputs: `archive/14_resnet50_agmfa_original.ipynb`;
старые веса, логи, predictions и запись experiments.csv не изменяются.


# ResNet50 + AGMFA — один эксперимент

[AGMFA paper](https://www.mdpi.com/2072-4292/13/16/3113), разделы 3.1–3.3.
Изменённый layer4 сохраняет 16×16. Три независимые channel-spatial attention masks
из выходов его bottleneck-блоков перемножаются и направляют fusion Res2/Res3/Res4.
Это не CBAM только поверх последнего слоя.

Наши адаптации: полный кадр 256×256; pretrained IMAGENET1K_V2; partial layer3/layer4;
замороженный backbone BatchNorm; новая fusion-нормализация GroupNorm(32,512).
В статье тип нормализации/ширина fusion явно не определены; здесь это фиксированные
настройки. Res4_1/2/3 — внутренние 512-канальные features после conv2/bn2/ReLU.
Финальный Res4 для fusion — отдельный полный 2048-канальный output третьего блока. Concat не требует одинакового количества каналов:
512+1024+2048 → 1×1 conv → 512 → ReLU → GroupNorm → GAP → dropout0.3 → FC20.

Robust AugMix+D4 сохранён без изменений, MixStyle выключен. CE, AdamW,
backbone LR5e-5/head LR5e-4, WD0.001, cosine25/eta_min1e-6, seed42.
25 эпох: первые 3 — warmup новых attention/fusion/head; далее также layer3/layer4.
Val только для оценки и выбора best checkpoint, без augmentation/TTA. **Test не читается.**
Обучение по умолчанию выключено; старые эксперименты и checkpoint не меняются.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import torch
from IPython.display import display

from archive.src.experiment import load_config, run_experiment
from archive.src.models.agmfa_checks import check_agmfa
from archive.src.utils import get_device, plot_history


In [ ]:
CONFIG = ROOT / 'configs/resnet50_agmfa_robust_augmix.json'
EXP_ID = 'resnet50_agmfa_robust_augmix_fixed_001'
RAW_DIR = REPOSITORY_ROOT / "data/raw"  # На Kaggle укажите каталог с train/, val/, classes.csv
DEVICE = 'auto'
BATCH_SIZE = 32  # Если не хватает GPU памяти, поставьте 16 перед запуском
NUM_WORKERS = 2
CPU_THREADS = 4
RUN_PREFLIGHT = True  # True: две synthetic проверки настоящего ResNet50, без данных
RUN_TRAINING = False   # True: один запуск обучения; preflight выполнится первым

device = get_device(DEVICE)
if device.type == 'cpu':
    torch.set_num_threads(CPU_THREADS)
cfg = load_config(CONFIG)
cfg['data'].update(raw_dir=str(RAW_DIR), batch_size=BATCH_SIZE,
                   num_workers=NUM_WORKERS if device.type == 'cuda' else 0)
cfg['training']['device'] = str(device)
print('Device:', device, '| experiment:', EXP_ID)
print('Artifacts:', ROOT / 'artifacts', '| train only, no test/TTA')


## Проверка до обучения
Проверка использует настоящую torchvision ResNet50 со случайными backbone весами:
скачивание ImageNet weights и чтение датасета не нужны. Два synthetic optimizer steps
проверяют размеры, product masks, градиенты всех attention, warmup, partial,
неизменность ранних весов и BN buffers, LR groups и deterministic validation.
При самом обучении загружаются pretrained IMAGENET1K_V2.


In [ ]:
if RUN_PREFLIGHT or RUN_TRAINING:
    preflight = check_agmfa(cfg, device=str(device), batch_size=2)
    print(json.dumps(preflight, indent=2))
else:
    print('Preflight выключен: RUN_PREFLIGHT = False для проверки.')


In [ ]:
if RUN_TRAINING:
    result = run_experiment(cfg, EXP_ID, project_root=ROOT,
                            show_plots=True, predict_test=False)
    print(f'Best epoch: {result.best_epoch}; val Accuracy: {result.val_accuracy:.4%}')
    print({name: str(path) for name, path in result.paths.items()})
else:
    print('Обучение выключено. Для одного запуска: RUN_TRAINING = False.')


## Сохранённый результат
Best/last checkpoint: `artifacts/models/<EXP_ID>/`.
History, summary, training.log и графики: `artifacts/logs/<EXP_ID>/`.
Val answers и `val_probabilities.npz`: `artifacts/predictions/<EXP_ID>/`.
После успешного запуска общий pipeline добавит запись в `experiments.csv`.
Следующая ячейка читает готовый результат, повторно ничего не обучает.
Сравнение с другими моделями выполняйте на той же обычной val, без TTA.


In [ ]:
logs = ROOT / 'artifacts/logs' / EXP_ID
if (logs / 'summary.json').is_file():
    summary = json.loads((logs / 'summary.json').read_text(encoding='utf-8'))
    display(pd.DataFrame([{'exp_id': EXP_ID, **summary}]))
    history = pd.read_csv(logs / 'history.csv')
    display(history.tail(3))
    if not RUN_TRAINING:
        plot_history(history, show=True)
else:
    print('Результат пока отсутствует. Выполните один запуск на Kaggle.')


In [ ]:
from archive.src.training.predict import load_probabilities
logs = ROOT/'artifacts/logs'/EXP_ID
predictions = ROOT/'artifacts/predictions'/EXP_ID
result = None
if not (logs/'summary.json').is_file():
    print('Новый recipe ещё не обучен. После Kaggle training выполните эту ячейку повторно.')
else:
    summary = json.loads((logs/'summary.json').read_text(encoding='utf-8'))
    display(pd.DataFrame([{'exp_id': EXP_ID, **summary}]))
    if result is None:
        plot_history(pd.read_csv(logs/'history.csv'), show=True)
    candidate = load_probabilities(predictions/'val_probabilities.npz').table(include_scores=True)
    new_ok = candidate.label.to_numpy() == candidate.true_label.to_numpy()
    rows = [{'model': 'AGMFA robust', 'correct': int(new_ok.sum()), 'accuracy': float(new_ok.mean())}]
    for name, run in [('stronger50', 'resnet50_partial_domain_shift_stronger_001'),
                      ('AGMFA original', 'resnet50_agmfa_robust_augmix_001')]:
        path = ROOT/'artifacts/predictions'/run/'val_probabilities.npz'
        if not path.is_file():
            continue
        old = load_probabilities(path).table(include_scores=True)
        if set(old.id) != set(candidate.id):
            raise ValueError('Val IDs отличаются; сравнение запрещено')
        old = old.set_index('id').loc[candidate.id].reset_index()
        if not (old.true_label.to_numpy() == candidate.true_label.to_numpy()).all():
            raise ValueError('Val labels отличаются; сравнение запрещено')
        old_ok = old.label.to_numpy() == old.true_label.to_numpy()
        rows.append({'model': name, 'correct': int(old_ok.sum()), 'accuracy': float(old_ok.mean()),
                     'fixed_by_robust': int((~old_ok & new_ok).sum()),
                     'new_errors_robust': int((old_ok & ~new_ok).sum())})
    display(pd.DataFrame(rows))

    baseline_path = ROOT / 'artifacts/predictions/resnet50_partial_domain_shift_stronger_001/val_probabilities.npz'
    if baseline_path.is_file():
        baseline = load_probabilities(baseline_path).table(include_scores=True)
        baseline = baseline.set_index('id').reindex(candidate.id).reset_index()
        if baseline.label.isna().any() or len(baseline) != len(candidate):
            raise ValueError('Baseline/candidate val IDs не совпадают')
        if not (baseline.true_label.to_numpy() == candidate.true_label.to_numpy()).all():
            raise ValueError('Baseline/candidate val targets не совпадают')
        old_ok = baseline.label.to_numpy() == baseline.true_label.to_numpy()
        new_ok = candidate.label.to_numpy() == candidate.true_label.to_numpy()
        print(f'Исправлено: {int((~old_ok & new_ok).sum())}; новых ошибок: {int((old_ok & ~new_ok).sum())}; '
                f'общих ошибок: {int((~old_ok & ~new_ok).sum())}')
        changed = pd.DataFrame({'id': candidate.id, 'true_label': candidate.true_label,
                                        'stronger50_label': baseline.label, 'agmfe_label': candidate.label,
                                        'stronger50_confidence': baseline.confidence, 'agmfe_confidence': candidate.confidence})
        display(changed[changed.stronger50_label != changed.agmfe_label].head(30))
    else:
        print('Baseline val_probabilities.npz отсутствует; сохранённый результат AGMFA показан выше.')
        
    print(f'Логи: {logs}')
    print(f'Веса: {ROOT/"artifacts/models"/EXP_ID}')
    print(f'Вероятности: {predictions/"val_probabilities.npz"}')
    print('Training row записана в experiments.csv; test inference не запускался.')


In [ ]:
from archive.src.training.predict import load_probabilities
logs = ROOT/'artifacts/logs'/EXP_ID
predictions = ROOT/'artifacts/predictions'/EXP_ID
result = None
if not (logs/'summary.json').is_file():
    print('Новый recipe ещё не обучен. После Kaggle training выполните эту ячейку повторно.')
else:
    summary = json.loads((logs/'summary.json').read_text(encoding='utf-8'))
    display(pd.DataFrame([{'exp_id': EXP_ID, **summary}]))
    if result is None:
        plot_history(pd.read_csv(logs/'history.csv'), show=True)
    candidate = load_probabilities(predictions/'val_probabilities.npz').table(include_scores=True)
    new_ok = candidate.label.to_numpy() == candidate.true_label.to_numpy()
    rows = [{'model': 'AGMFA robust', 'correct': int(new_ok.sum()), 'accuracy': float(new_ok.mean())}]
    for name, run in [('stronger50', 'resnet50_partial_domain_shift_stronger_001'),
                      ('AGMFA stronger', 'resnet50_agmfa_robust_augmix_fixed_001')]:
        path = ROOT/'artifacts/predictions'/run/'val_probabilities.npz'
        if not path.is_file():
            continue
        old = load_probabilities(path).table(include_scores=True)
        if set(old.id) != set(candidate.id):
            raise ValueError('Val IDs отличаются; сравнение запрещено')
        old = old.set_index('id').loc[candidate.id].reset_index()
        if not (old.true_label.to_numpy() == candidate.true_label.to_numpy()).all():
            raise ValueError('Val labels отличаются; сравнение запрещено')
        old_ok = old.label.to_numpy() == old.true_label.to_numpy()
        rows.append({'model': name, 'correct': int(old_ok.sum()), 'accuracy': float(old_ok.mean()),
                     'fixed_by_robust': int((~old_ok & new_ok).sum()),
                     'new_errors_robust': int((old_ok & ~new_ok).sum())})
    display(pd.DataFrame(rows))

    baseline_path = ROOT / 'artifacts/predictions/resnet50_partial_domain_shift_stronger_001/val_probabilities.npz'
    if baseline_path.is_file():
        baseline = load_probabilities(baseline_path).table(include_scores=True)
        baseline = baseline.set_index('id').reindex(candidate.id).reset_index()
        if baseline.label.isna().any() or len(baseline) != len(candidate):
            raise ValueError('Baseline/candidate val IDs не совпадают')
        if not (baseline.true_label.to_numpy() == candidate.true_label.to_numpy()).all():
            raise ValueError('Baseline/candidate val targets не совпадают')
        old_ok = baseline.label.to_numpy() == baseline.true_label.to_numpy()
        new_ok = candidate.label.to_numpy() == candidate.true_label.to_numpy()
        print(f'Исправлено: {int((~old_ok & new_ok).sum())}; новых ошибок: {int((old_ok & ~new_ok).sum())}; '
                f'общих ошибок: {int((~old_ok & ~new_ok).sum())}')
        changed = pd.DataFrame({'id': candidate.id, 'true_label': candidate.true_label,
                                        'stronger50_label': baseline.label, 'agmfe_label': candidate.label,
                                        'stronger50_confidence': baseline.confidence, 'agmfe_confidence': candidate.confidence})
        display(changed[changed.stronger50_label != changed.agmfe_label].head(30))
    else:
        print('Baseline val_probabilities.npz отсутствует; сохранённый результат AGMFA показан выше.')
        
    print(f'Логи: {logs}')
    print(f'Веса: {ROOT/"artifacts/models"/EXP_ID}')
    print(f'Вероятности: {predictions/"val_probabilities.npz"}')
    print('Training row записана в experiments.csv; test inference не запускался.')


In [ ]:
from pathlib import Path
import shutil
import zipfile

ARTIFACTS = Path("/kaggle/working/project/artifacts")
EXP_ID = "resnet50_agmfa_robust_augmix_fixed_001"
OUTPUT_DIR = Path("/kaggle/working")

# logs и predictions как раньше
targets = {
    "logs": ARTIFACTS / "logs" / EXP_ID,
    "predictions": ARTIFACTS / "predictions" / EXP_ID,
}

for name, folder in targets.items():
    if not folder.exists():
        print(f"SKIP: {folder} не существует")
        continue

    output_base = OUTPUT_DIR / f"{EXP_ID}_{name}"

    shutil.make_archive(
        str(output_base),
        "zip",
        root_dir=folder
    )

    print(f"Created: {output_base}.zip")


# models отдельно: best и last
models_dir = ARTIFACTS / "models" / EXP_ID

if models_dir.exists():
    for model_type in ["best", "last"]:
        files = [
            p for p in models_dir.rglob("*")
            if p.is_file() and model_type.lower() in p.name.lower()
        ]

        if not files:
            print(f"SKIP: model '{model_type}' не найден")
            continue

        zip_path = OUTPUT_DIR / f"{EXP_ID}_{model_type}.zip"

        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
            for file in files:
                z.write(
                    file,
                    arcname=file.relative_to(models_dir)
                )

        print(f"Created: {zip_path}")
else:
    print(f"SKIP: {models_dir} не существует")